In [ ]:
# Environment configuration — replace the placeholder in the Databricks widget.
dbutils.widgets.text("storage_account", "<YOUR_STORAGE_ACCOUNT>")
storage_account = dbutils.widgets.get("storage_account")
if not storage_account or storage_account.startswith("<"):
    raise ValueError("Set the storage_account widget before running.")


In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import time

### Data Transformation

In [ ]:
df = spark.read.format("parquet")\
    .load(f"abfss://bronze@{storage_account}.dfs.core.windows.net/customers")
df.display()

### Dropping unnecessary col

In [ ]:
df = df.drop('_rescued_data')
df.display()

### From Emails let's retrieve the domain name only

In [ ]:
df = df.withColumn("domains", split(col('email'), '@')[1])
df.display()

### By using Aggregation Func let's count how many customers have which top email domain

In [ ]:
df.groupby("domains").agg(count("customer_id").alias("total_customers")).sort("total_customers",ascending=False).display()

### Now we know the top 3 domains are: Gmail, Hotmail, Yahoo

### Now let's filter the data based on these three domains

In [ ]:
df_gmail = df.filter(col("domains") == "gmail.com")
df_gmail.display()
time.sleep(5)

df_hotmail = df.filter(col("domains") == "hotmail.com")
df_hotmail.display()
time.sleep(5)

df_yahoo = df.filter(col("domains") == "yahoo.com")
df_yahoo.display()
time.sleep(5)

### Next Transformation: Concatenate the First & Last Name: Full Name

In [ ]:
df = df.withColumn("full_name",concat(col("first_name"),lit(" "),col("last_name")))
df = df.drop("first_name","last_name")
df.display()

### Creating a Schema for Silver from here:

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS databricks_etl_catalog.silver.customer_silver
USING DELTA
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/customers'

In [ ]:
%sql
Select * 
FROM databricks_etl_catalog.silver.customer_silver

### Data Writing to Delta Live Tables in Azure Data Lake Gen 2 Storage

In [ ]:
df.write.mode("overwrite").format("delta").save(f"abfss://silver@{storage_account}.dfs.core.windows.net/customers")